# 04 · Fashion retrieval with CLIP
Compare a pretrained image–text model with keyword ranking on the same catalogue sample. Use the full project folder. For the model cell, use a notebook environment with `torch`, `transformers>=4.46,<5`, `Pillow`, `requests` and internet access to download the weights and public product images.

In [1]:
from pathlib import Path
import sys

# Start Jupyter in the project folder or its notebooks folder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend/catalogue.json").is_file()), None)
if root is None:
    raise RuntimeError("Open the project directory first. In Colab, upload/mount the project and %cd into it.")
sys.path.insert(0, str(root / "backend"))
import json, random, re
query = "black shirt"
rows = json.loads((root / "backend/catalogue.json").read_text())
items = {}
for name, tags, tip, *extra in rows:
    if extra and extra[0].get("image_url", "").startswith("https://") and extra[0].get("product_url"):
        items[extra[0]["product_url"]] = (name, tags, extra[0])
sample = random.Random(42).sample(list(items.values()), min(100, len(items)))
print(f"{len(sample)} sampled from {len(items)} linked listings with images.")

100 sampled from 4998 linked listings with images.


### Encode catalogue images and search
Set `run_clip = True` to download the model and encode the sample. Failed image downloads are counted. Re-run only the final cell to search the same images with a new query.

In [2]:
run_clip = False
if run_clip:
    import io, requests, torch
    from PIL import Image, UnidentifiedImageError
    from transformers import CLIPModel, CLIPProcessor
    import torch.nn.functional as F

    checkpoint = "openai/clip-vit-base-patch32"
    model = CLIPModel.from_pretrained(checkpoint).eval()
    processor = CLIPProcessor.from_pretrained(checkpoint)
    encoded, vectors = [], []
    with requests.Session() as session, torch.inference_mode():
        for item in sample:
            try:
                response = session.get(item[2]["image_url"], timeout=10)
                response.raise_for_status()
                with Image.open(io.BytesIO(response.content)) as img:
                    inputs = processor(images=img.convert("RGB"), return_tensors="pt")
            except (OSError, requests.RequestException, Image.DecompressionBombError):
                continue
            vectors.append(F.normalize(model.get_image_features(**inputs), dim=-1))
            encoded.append(item)
    print(f"Encoded {len(encoded)}/{len(sample)} images.")
    if not vectors:
        raise RuntimeError("No sample images could be downloaded.")
    image_vectors = torch.cat(vectors)

In [3]:
if run_clip:
    with torch.inference_mode():
        inputs = processor(text=["a product photo of " + query], return_tensors="pt", truncation=True)
        text_vector = F.normalize(model.get_text_features(**inputs), dim=-1)
        similarity = (image_vectors @ text_vector.T).flatten()
    words = set(re.findall(r"\w+", query.casefold()))
    keyword_order = sorted(range(len(encoded)), key=lambda i: len(words & set(encoded[i][1].split())), reverse=True)
    for method, indices in [("CLIP", similarity.argsort(descending=True).tolist()), ("Keyword overlap", keyword_order)]:
        print(method)
        for rank, i in enumerate(indices[:10], 1):
            print(rank, encoded[i][0], encoded[i][2]["product_url"])
else:
    print("Set run_clip to True to run image retrieval.")

Set run_clip to True to run image retrieval.


For each query, independently mark each displayed listing relevant or irrelevant using the requested garment, colour and other explicit preferences. Use the same judgements for both lists. With 10 results, Precision@10 is relevant results divided by 10. Report how many images were encoded and how many queries were judged.